In [1]:
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
from typing import Optional, Type, Union
from scipy.interpolate import CubicSpline
import time

#few imports
from few.trajectory.inspiral import EMRIInspiral #function to generate trajectories
from few.trajectory.ode.base import ODEBase #superclass for defining modified flux trajectories
from few.trajectory.ode.flux import KerrEccEqFlux #Adiabatic KerrEccEq base trajectory class

from few.amplitude.ampinterp2d import AmpInterpKerrEccEq
from few.summation.interpolatedmodesum import InterpolatedModeSum
from few.utils.modeselector import ModeSelector

from few.waveform import (
    FastSchwarzschildEccentricFlux, 
    SlowSchwarzschildEccentricFlux, 
    Pn5AAKWaveform,
    FastKerrEccentricEquatorialFlux,
    GenerateEMRIWaveform,
)

##### SuperKludge Imports #####
from few.trajectory.ode.flux import SuperKludgeFlux #trajectory module
from few.waveform.waveform import SuperKludgeWaveform #waveform module
###############################

from few.waveform.base import SphericalHarmonicWaveformBase #generic waveform generation class
from few.utils.baseclasses import SphericalHarmonic, KerrEccentricEquatorial, ParallelModuleBase, BackendLike

from few.utils.constants import YRSID_SI

from few.utils.utility import ( 
    get_mismatch, 
    get_m2_at_t, 
    get_p_at_t, 
    )

from few.utils.geodesic import (
    get_fundamental_frequencies,
    get_separatrix,
    get_kerr_geo_constants_of_motion,
    ELQ_to_pex,
    )
from few.trajectory.inspiral import get_0PA_frequencies

#lisa-on-gpu import
from fastlisaresponse import ResponseWrapper  # Response function 
import importlib
import utils
importlib.reload(utils)
from utils import (
    compute_fft_with_windowing,
    inner_product_from_fft,
    calculate_overlap_1pa_vs_2pa,
    calculate_optimal_snr_1pa_vs_2pa,
    calculate_overlap_0pa_vs_2pa,
    calculate_optimal_snr_0pa_vs_2pa
) #mx-Liu

#SEF imports
from stableemrifisher.utils import generate_PSD, padding, inner_product

#few imports
from few.trajectory.inspiral import EMRIInspiral #function to generate trajectories
from few.trajectory.ode.flux import KerrEccEqFlux #Adiabatic KerrEccEq base trajectory class

from few.waveform import (
    FastKerrEccentricEquatorialFlux,
    GenerateEMRIWaveform,
)

#lisa-on-gpu import
from fastlisaresponse import ResponseWrapper  # Response function 
#LISAanalysistools imports
from lisatools.detector import ESAOrbits, EqualArmlengthOrbits #ESAOrbits correspond to esa-trailing-orbits.h5, EqualArmlengthOrbits are equalarmlength-orbits.h5
from lisatools.sensitivity import get_sensitivity, A1TDISens, E1TDISens, T1TDISens

import numpy as np
from scipy.optimize import fsolve
from few.trajectory.inspiral import get_0PA_frequencies

SK_traj_0PA = EMRIInspiral(func=KerrEccEqFlux)

# 2PA: SuperKludge flux with post-adiabatic corrections
SK_traj_2PA = EMRIInspiral(func=SuperKludgeFlux)

def find_p0_for_target_gw_freq(m1, m2, a, e0, x, target_f_gw=1e-3):
    """
    Find p0 value that gives target gravitational wave frequency
    
    Args:
        m1, m2: masses in solar masses
        a: dimensionless spin
        e0: initial eccentricity 
        x: cos(inclination)
        target_f_gw: target GW frequency in Hz (default 1e-3)
    
    Returns:
        p0: initial semi-latus rectum
    """
    
    def freq_residual(p0):
        # Get orbital frequencies
        f_phi, f_theta, f_r = get_0PA_frequencies(m1, m2, a, p0, e0, x)
        
        # For (2,2) mode: f_GW ≈ 2 * f_phi
        f_gw_calculated = 2 * f_phi
        
        return f_gw_calculated - target_f_gw
    
    # Initial guess (larger p0 gives lower frequency)
    p0_guess = 15.0
    
    # Solve for p0
    p0_solution = fsolve(freq_residual, p0_guess)[0]
    
    # Verify the result
    f_phi, f_theta, f_r = get_0PA_frequencies(m1, m2, a, p0_solution, e0, x)
    f_gw_check = 2 * f_phi
    
    print(f"Target GW frequency: {target_f_gw:.2e} Hz")
    print(f"Calculated GW frequency: {f_gw_check:.2e} Hz")
    print(f"Required p0: {p0_solution:.3f}")
    
    return p0_solution

use_gpu = False #False if your computer sucks (mine does)

if use_gpu:
    import cupy as xp
    from cupy import fft as fft_module
else:
    import numpy as xp
    from numpy import fft as fft_module
    
if not use_gpu:
    
    import few
    
    #tune few configuration
    cfg_set = few.get_config_setter(reset=True)
    
    cfg_set.enable_backends("cpu")
    cfg_set.set_log_level("info");
else:
    pass #let the backend decide for itself

#initializing trajectories
#start = time.time()
#SK_traj = EMRIInspiral(func=SuperKludgeFlux)
#print('time in loading SK_traj = EMRIInspiral(func=SuperKludgeFlux)',time.time() - start)

#waveform setup
sum_kwargs=dict(pad_output=False, odd_len=True)
waveform_model = GenerateEMRIWaveform(SuperKludgeWaveform,sum_kwargs=sum_kwargs, return_list=False)
SK_traj_2PA = EMRIInspiral(func=SuperKludgeFlux)











In [2]:
pa2_snr = 20
use_emri = True
if use_emri:
    # EMRI define parameters
    m1 = 1e6
    m2 = 1e1
    a = 0.99
    e0 = 0.8
    Y0 = 1.0
    dist = 4.027199942062994#1.0#*58.09/20
    qS = np.pi/3
    phiS = np.pi/4
    qK = np.pi/5.
    phiK = np.pi/6.
    Phi_phi0 = 0.1
    Phi_theta0 = 0.2
    Phi_r0 = 0.3
    dt = 10.0
    T = 2.5#2#1.0
    chi2 = -0.9#0.971 #dimensionless secondary spin
    #p0 = 9.389272347381176 + 0.1
    #p0 = get_p_at_t(traj_module=SK_traj_2PA, t_out=T,  traj_args=[m1, m2, a, e0, Y0, *add_args]) + 0.1 

else:
    #IMRI params:
    m1 = 1e6
    m2 = 1e4
    a = 0.99
    e0 = 0.8#0.5#0.3
    Y0 = 1.0
    dist = 1.8780649822451412#1.0#*58.09
    qS = np.pi/3
    phiS = np.pi/4
    qK = np.pi/5.
    phiK = np.pi/6.
    Phi_phi0 = 0.1
    Phi_theta0 = 0.2
    Phi_r0 = 0.3
    dt = 10.0
    T = 0.25#2#1.0
    chi2 = 0.9
    #p0 = 49.47637894374171 + 0.1





evolve_1PA = True
evolve_primary = False
evolve_2PA = True
add_args = [chi2, evolve_1PA, evolve_primary, evolve_2PA]
p0 = get_p_at_t(traj_module=SK_traj_2PA, t_out=T,  traj_args=[m1, m2, a, e0, Y0, *add_args]) + 0.1 

theta = np.array([m1, m2, a, p0, e0])
theta_inj_2PA = np.array([m1, m2, a, p0, e0, Y0])


m1_2pa, m2_2pa, a_2pa, p0_2pa, e0_2pa, Y0_2pa = theta_inj_2PA
# Generate 2PA trajectory with additional parameters
evolve_1PA = True
evolve_primary = False  
evolve_2PA = True
add_args_2pa = [chi2, evolve_1PA, evolve_primary, evolve_2PA]

traj_2PA = SK_traj_2PA.get_inspiral(
    m1_2pa, m2_2pa, a_2pa, p0_2pa, e0_2pa, Y0_2pa, *add_args_2pa,
    T=T, dt=dt, err=1e-11, DENSE_STEPPING=False,
    buffer_length=1000, integrate_backwards=False,
    max_step_size=None
)
t_2PA, p_2PA, e_2PA, x_2PA = traj_2PA[0], traj_2PA[1], traj_2PA[2], traj_2PA[3]
Omega_phi_2PA, Omega_theta_2PA, Omega_r_2PA = get_fundamental_frequencies(
    a_2pa, p_2PA, e_2PA, x_2PA
)      
    
from scipy.integrate import cumulative_trapezoid
from few.utils.constants import MTSUN_SI
import numpy as np



#wave = waveform_response(*wave_params, **emri_kwargs)

# with longer signals we care less about this
t0 = 10000.0  # throw away on both ends when our orbital information is weird
#initialize LISA response model
tdi_gen ="2nd generation"# "1st generation"#
order = 20  # interpolation order (should not change the result too much)
tdi_kwargs_esa = dict(
    orbits=EqualArmlengthOrbits(use_gpu=use_gpu),
    order=order, 
    tdi=tdi_gen, 
    tdi_chan="AET",
)
index_lambda = 8 #azimuthal sky location (phiS)
index_beta = 7 #polar sky location (qS)
waveform_response = ResponseWrapper(
                        waveform_gen=waveform_model, #the waveform model around which to wrap the Response
                        Tobs=T,
                        t0=t0,
                        dt=dt,
                        index_lambda=index_lambda,
                        index_beta=index_beta,
                        flip_hx=True,  # set to True if waveform is h+ - ihx (FEW is)
                        use_gpu=use_gpu,
                        is_ecliptic_latitude=False,  # False if using polar angle (theta)
                        remove_garbage="zero",  # removes the beginning of the signal that has bad information
                        **tdi_kwargs_esa,
                        )
#noise model
channels = [A1TDISens, E1TDISens, T1TDISens] #AET TDI channels. This should be the same as the channels used in ResponseWrapper init
noise_kwargs = [{"sens_fn": channel_i} for channel_i in channels] #Keyword argument for each channel to be supplied to get_sensitivity inside SEF



In [3]:
# Create common time grid for interpolation
t_min = t_2PA.min()
t_max = t_2PA.max()
t_common = np.linspace(t_min, t_max, 1000)

# Interpolate Omega_phi onto common time grid
from scipy.interpolate import CubicSpline

# Create splines for both trajectories
spline_2PA = CubicSpline(t_2PA, Omega_phi_2PA)

# Evaluate on common time grid
Omega_phi_2PA_interp = spline_2PA(t_common)

m_mode=2
Msec = (m1 + m2) * MTSUN_SI  # s
# 3)  2PA reference freq：f_gw(t) = m * Omega_SI / (2π)
Omega2_SI = Omega_phi_2PA_interp / Msec          # rad/s
f_gw = m_mode * Omega2_SI / (2.0 * np.pi)        # Hz

# 4) ：w(t) = Σ 1/S_n(f)
w = np.zeros_like(f_gw)



for ch in channels:
    S_n = get_sensitivity(f_gw, sens_fn=ch)      # 1/Hz
    S_n = np.maximum(S_n, 1e-60)
    w += 1.0 / S_n
        
def weighted_phase_metric_dimlessOmega(
    t_common,
    Omega_phi_0PA_interp,   # （dimensionless, ~1/M）
    Omega_phi_2PA_interp,   #     
):



    # 1) ΔΩ in SI：rad/s
    dOmega_geo = Omega_phi_0PA_interp - Omega_phi_2PA_interp
    dOmega_SI  = dOmega_geo / Msec  # rad/s

    # 2) Δφ_gw(t) = m ∫ ΔΩ_SI dt
    dphi = np.concatenate([[0.0], m_mode * cumulative_trapezoid(dOmega_SI, t_common)])  # rad

    num = np.trapz(w * dphi**2, t_common)
    den = np.trapz(w, t_common)
    return np.sqrt(num / den)

def compute_omega_phi_difference(theta, Omega_phi_2PA):
    """
    Compare Omega_phi between 0PA and 2PA trajectories
    
    Args:
        theta: [m1, m2, a, p0, e0, Y0] for 0PA trajectory
        theta_inj_2PA: [m1, m2, a, p0, e0, Y0] for 2PA trajectory  
        dt: time step in seconds
        T: observation time in years
        chi2: secondary spin parameter
        
    Returns:
        scalar difference between interpolated Omega_phi trajectories
    """
    
    # Unpack parameters
    m1_0pa, m2_0pa, a_0pa, p0_0pa, e0_0pa = theta
    
    
    evolve_1PA = False  
    evolve_primary = False  
    evolve_2PA = False  
    add_args = [chi2, evolve_1PA, evolve_primary, evolve_2PA]    
    
    # Generate 0PA trajectory
    traj_0PA = SK_traj_0PA.get_inspiral(
        m1_0pa, m2_0pa, a_0pa, p0_0pa, e0_0pa, Y0, *add_args,
        T=T, dt=dt, err=1e-11, DENSE_STEPPING=False,
        buffer_length=1000, integrate_backwards=False,
        max_step_size=None
    )
    

    # Extract trajectory components
    # Format: (t, p, e, x, Phi_phi, Phi_theta, Phi_r)
    t_0PA, p_0PA, e_0PA, x_0PA = traj_0PA[0], traj_0PA[1], traj_0PA[2], traj_0PA[3]
  
    
    # Calculate fundamental frequencies using geodesic functions
    Omega_phi_0PA, Omega_theta_0PA, Omega_r_0PA = get_fundamental_frequencies(
        a_0pa, p_0PA, e_0PA, x_0PA
    )
    

    
    # Create common time grid for interpolation
    #t_min = max(t_0PA.min(), t_2PA.min())
    #t_max = min(t_0PA.max(), t_2PA.max())
    #t_common = np.linspace(t_min, t_max, 1000)
    
    # Interpolate Omega_phi onto common time grid    
    # Create splines for both trajectories
    spline_0PA = CubicSpline(t_0PA, Omega_phi_0PA)
    #spline_2PA = CubicSpline(t_2PA, Omega_phi_2PA)
    
    # Evaluate on common time grid
    Omega_phi_0PA_interp = spline_0PA(t_common)
    #Omega_phi_2PA_interp = spline_2PA(t_common)
    
    # Calculate difference (RMS or mean absolute difference)
    diff_array = Omega_phi_0PA_interp - Omega_phi_2PA_interp
    
    # Return scalar difference (RMS)
    #scalar_diff = np.sqrt(np.mean(diff_array**2))
    scalar_diff = weighted_phase_metric_dimlessOmega(
        t_common,
        Omega_phi_0PA_interp,
        Omega_phi_2PA_interp,
    )    
    
    return scalar_diff

from scipy.optimize import minimize

def optimize_0pa_params(initial_guess):
    """
    Use Nelder-Mead to find best-fit 0PA parameters that minimize Omega_phi difference with 2PA
    """
    #if initial_guess is None:
    #    initial_guess = theta_inj_2PA.copy()  # Start with 2PA params
    #    #initial_guess = np.array([m1*0.999, m2, a, p0, e0, Y0])
    
    # Objective function to minimize
    def objective(theta):
        try:
            func_val = compute_omega_phi_difference(theta, Omega_phi_2PA)
            #print(f"Params: {repr(theta)}, Func value: {func_val:.6e}")
            return func_val
        except Exception as e:
            print(f"Params: {theta}, ERROR: {str(e)}")
            return 1e10  # Return large value if trajectory fails
    
    # Optimize using Nelder-Mead
    result = minimize(objective, initial_guess, method='Nelder-Mead', 
                     options={'maxiter': 1000, 'fatol': 1e-4})
    
    return result

# Run optimization
result = optimize_0pa_params(theta_inj_2PA[:5])
print(f"Best-fit 0PA params: {result.x}")
print(f"Minimum frequency difference: {result.fun}")
print(f"Success: {result.success}")    

/tmp/ipykernel_335311/2888274576.py:48: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  num = np.trapz(w * dphi**2, t_common)
/tmp/ipykernel_335311/2888274576.py:49: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  den = np.trapz(w, t_common)


Params: [1.00000000e+06 1.00000000e+01 1.03950000e+00 8.07370829e+00
 8.00000000e-01], ERROR: Interpolation: a out of bounds. Must be between -0.999 and 0.999.
Params: [1.00000000e+06 1.00000000e+01 9.90000000e-01 8.07370829e+00
 8.40000000e-01], ERROR: Interpolation: p 8.073708289753787 out of bounds. Must be between 8.964147909460213 and 200.
Params: [1.02000000e+06 1.02000000e+01 1.00980000e+00 8.23518246e+00
 7.60000000e-01], ERROR: Interpolation: a out of bounds. Must be between -0.999 and 0.999.
Params: [1.00500000e+06 1.00500000e+01 9.94950000e-01 8.11407683e+00
 8.20000000e-01], ERROR: Interpolation: p 8.114076831202556 out of bounds. Must be between 8.279550876857686 and 200.
Params: [1.00000000e+06 1.00000000e+01 1.01475000e+00 8.07370829e+00
 8.00000000e-01], ERROR: Interpolation: a out of bounds. Must be between -0.999 and 0.999.
Params: [1.00000000e+06 1.00000000e+01 9.90000000e-01 8.07370829e+00
 8.20000000e-01], ERROR: Interpolation: p 8.073708289753787 out of bounds. Mu

In [5]:
#Minimum frequency difference: 0.016327457207079593

In [6]:
def log_reward(params):
    """
    Calculate optimal SNR reward
    
    Parameters:
    -----------
    params : array-like, shape (N_fiducial, 6) or (6,)
        Parameter array(s) [m1, m2, a, p0, e0, chi2]
        Can handle single parameter set or batch of parameter sets
        
    Returns:
    --------
    reward : float or array
        Optimal SNR(s), or 0 if calculation fails
    """
    params = np.asarray(params)
    
    # Handle both single parameter set and batch
    if False:#params.ndim == 1:
        # Single parameter set
        #m1_in, m2_in, a_in, p0_in, e0_in = params #0pa not need chi2
        try:
            #result = optimize_0pa_params(params)

            #m1_in, m2_in, a_in, p0_in, e0_in = result.x
            #optimal_snr = calculate_optimal_snr_0pa_vs_2pa(
            #    m1_in, m2_in, a_in, p0_in, e0_in, Y0, dist, qS, phiS, qK, phiK,
            #    Phi_phi0, Phi_theta0, Phi_r0, chi2,
            #    waveform_response=waveform_response, PSD=PSD_funcs, dt=dt, T=T, N_fiducial=N_fiducial,
            #    waveform_2pa_fft=waveform_2PA_fid_fft, xp=xp, use_gpu=use_gpu
            #    )
#
            ##print(optimal_snr,'param',params)
            #return float(optimal_snr)
            #return result.fun
            
            freq_diff = compute_omega_phi_difference(params, Omega_phi_2PA)
            res = -np.log(freq_diff)
            
            return res
        except KeyboardInterrupt:
            # If keyboard interrupt, re-raise to terminate
            raise
        except:
            # For any other exception, return -np.inf
            #print('error 1 in param',repr(params))
            return float(-np.inf)
    else:
        # Batch of parameter sets
        rewards = np.zeros(params.shape[0])
        for i, param_set in enumerate(params):
            #m1_in, m2_in, a_in, p0_in, e0_in = param_set
            try:
                #result = optimize_0pa_params(param_set)
                #m1_in, m2_in, a_in, p0_in, e0_in = result.x
                #optimal_snr = calculate_optimal_snr_0pa_vs_2pa(
                #    m1_in, m2_in, a_in, p0_in, e0_in, Y0, dist, qS, phiS, qK, phiK,
                #    Phi_phi0, Phi_theta0, Phi_r0, chi2,
                #    waveform_response=waveform_response, PSD=PSD_funcs, dt=dt, T=T, N_fiducial=N_fiducial,
                #    waveform_2pa_fft=waveform_2PA_fid_fft, xp=xp, use_gpu=use_gpu                  
                #)
                #rewards[i] = optimal_snr
                #rewards[i] = result.fun
                if param_set[2] > 0.999:
                    rewards[i] = -np.inf
                else:
                    freq_diff = compute_omega_phi_difference(param_set, Omega_phi_2PA)
                    #print(freq_diff)
                    res = -np.log(freq_diff)*10  
                    rewards[i] = res
            except KeyboardInterrupt:
                # If keyboard interrupt, re-raise to terminate
                raise
            except:
                # For any other exception, set to -np.inf
                print('error 2 in param',repr(param_set))
                rewards[i] = -np.inf
        return rewards

# Reference values
ref_vals = np.array([m1, m2, a, p0, e0])
def prior_transform(u):
    """
    Transform uniform samples u from (0,1) to parameter space
    Each parameter varies in range [old_val*(1-0.1), old_val*(1+0.1)]
    
    Parameters:
    -----------
    u : array-like, shape (6,)
        Uniform samples from (0,1) for [m1, m2, a, p0, e0, chi2]
        
    Returns:
    --------
    params : array-like, shape (6,)
        Transformed parameters [m1, m2, a, p0, e0, chi2]
    """

    
    # Transform each parameter: old_val*(1-0.1) + u*(old_val*0.2)
    # This gives range [old_val*0.9, old_val*1.1]
    params = ref_vals * (0.9 + u * 0.2)
    
    return params

def inverse_prior_transform(params):
    """
    Transform parameters back to uniform space [0,1]
    
    Parameters:
    -----------
    params : array-like, shape (5,)
        Parameters [m1, m2, a, p0, e0]
        
    Returns:
    --------
    u : array-like, shape (5,)
        Uniform samples in (0,1)
    """
    u = (params / ref_vals - 0.9) / 0.2
    return u
    
external_lhs_points = np.array([[0.5,0.5,0.5,0.5,0.5]])
perturbed_points = []
for i in range(1000):
    if i == 0:
        perturbed = external_lhs_points[0]
        #perturbed = external_lhs_points[0] + np.random.normal(0, 1e-4, 5)
    else:
        perturbed = external_lhs_points[0] + np.random.normal(0, 1e-4, 5)#1e-4 IMRI
        #perturbed = np.array([0.50002825, 0.50005415, 0.49995162, 0.50005497, 0.49998339])
    perturbed_points.append(perturbed)
external_lhs_points = np.array(perturbed_points)

#external_lhs_points = np.array([inverse_prior_transform(np.array([9.99999868e+05, 9.99997969e+00, 9.90020844e-01, 8.07371883e+00, 8.00001391e-01]))]) #a=0.99,e=0.8 EMRI
#external_lhs_points = np.array([inverse_prior_transform(np.array([9.99961290e+05, 9.99996529e+00, 9.89987083e-01, 8.75785416e+00, 5.00007806e-01]))])#a=0.99,e=0.5 EMRI
#param_array = np.load('traj_opt_param.npy')[:,:5]
#external_lhs_points = inverse_prior_transform(param_array)
external_lhs_log_rewards = log_reward(prior_transform(external_lhs_points))


#def main():

external_lhs_log_rewards.max(),#param_array[0]

/tmp/ipykernel_335311/2888274576.py:48: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  num = np.trapz(w * dphi**2, t_common)
/tmp/ipykernel_335311/2888274576.py:49: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  den = np.trapz(w, t_common)


(np.float64(9.132619564532522),)

In [ ]:

import sys
modules_to_clear = [name for name in sys.modules if name.startswith('parismc')]
for module_name in modules_to_clear:
    if module_name in sys.modules:
        del sys.modules[module_name]

import os
import numpy as np  
import parismc

if True:
    """
    Main function to run the multimodal sampling example.
    """
    print("Multimodal 10D Sampling Example")
    print("===============================")
    
    # Problem setup
    ndim = 5
    n_seed = 20  # Number of walkers/chains
    sigma = 1e-6#0.01  # Initial covariance scale
    savepath = './local_paris_results/'
    
    # Create save directory
    os.makedirs(savepath, exist_ok=True)
    
    # Initialize covariance matrices for each walker
    init_cov_list = []
    for i in range(n_seed):
        init_cov_list.append(sigma**2 * np.eye(ndim))
    
    # Configure sampler for challenging multimodal problem
    config = parismc.SamplerConfig(
        proc_merge_prob=0.9,           # High probability for merging similar chains
        alpha=1000,                    # Importance sampling parameter
        latest_prob_index=1000,        # Use recent samples for weighting
        trail_size=int(1e3),          # Maximum trials per iteration
        boundary_limiting=True,        # Enable boundary constraints
        use_beta=True,                # Use beta correction for boundaries
        integral_num=int(1e5),        # MC samples for beta estimation
        gamma=100,                    # Covariance update frequency
        exclude_scale_z=np.inf,       # No exclusion based on weights
        use_pool=False,               # Set to True for multiprocessing
        n_pool=36                      # Number of processes (if use_pool=True)
    )
    
    print(f"Problem dimension: {ndim}")
    print(f"Number of walkers: {n_seed}")
    print(f"Initial covariance scale: {sigma}")
    print(f"Save path: {savepath}")
    print(f"Multiprocessing: {config.use_pool}")
    
    # Initialize sampler
    print("\nInitializing sampler...")
    sampler = parismc.Sampler(
        ndim=ndim, 
        n_seed=n_seed,
        log_reward_func=log_reward,
        init_cov_list=init_cov_list,
        prior_transform=prior_transform,
        config=config
    )
    
    # Prepare initial samples using Latin Hypercube Sampling
    print("Preparing LHS samples...")
    #sampler.prepare_lhs_samples(lhs_num=int(1e5), batch_size=100)
    
    # Run the sampling process
    print("Starting sampling process...")
    print("(This may take several minutes for 10,000 iterations)")
    
    try:
        sampler.run_sampling(
            num_iterations=10000, 
            savepath=savepath,
            print_iter=100,  # Print progress every 100 iterations
            external_lhs_points=external_lhs_points,
            external_lhs_log_rewards=external_lhs_log_rewards        
        )
        
        print("\nSampling completed successfully!")
        
        # Analyze results
        #analyze_results(sampler, savepath)
        
        # Create visualizations
        #visualize_marginal_distributions(sampler, savepath)
        
    except KeyboardInterrupt:
        print("\nSampling interrupted by user.")
        print("Partial results have been saved.")
        #if hasattr(sampler, 'searched_points_list'):
        #    analyze_results(sampler, savepath)
        #    try:
        #        visualize_marginal_distributions(sampler, savepath)
        #    except:
        #        print("Could not create visualizations with partial results.")
    
    except Exception as e:
        print(f"\nError during sampling: {e}")
        raise
    
    print(f"\nResults saved to: {savepath}")
    print("Example completed!")

#if __name__ == "__main__":
#    main()

Multimodal 10D Sampling Example
Problem dimension: 5
Number of walkers: 20
Initial covariance scale: 1e-06
Save path: ./local_paris_results/
Multiprocessing: False

Initializing sampler...
Preparing LHS samples...
Starting sampling process...
(This may take several minutes for 10,000 iterations)


Sampling:   0%|          | 0/9999 [00:00<?, ?it/s]

/tmp/ipykernel_335311/2888274576.py:48: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  num = np.trapz(w * dphi**2, t_common)
/tmp/ipykernel_335311/2888274576.py:49: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  den = np.trapz(w, t_common)
/home/svu/e1101894/PARIS/PARIS_package/parismc/parismc/sampler.py:611: RuntimeWarning: divide by zero encountered in log
  logZ = c_term - np.log(Nsum) + np.log(wsum)
/home/svu/e1101894/PARIS/PARIS_package/parismc/parismc/sampler.py:611: RuntimeWarning: invalid value encountered in scalar add
  logZ = c_term - np.log(Nsum) + np.log(wsum)


In [19]:
compute_omega_phi_difference(theta, Omega_phi_2PA)

/tmp/ipykernel_3529932/2888274576.py:48: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  num = np.trapz(w * dphi**2, t_common)
/tmp/ipykernel_3529932/2888274576.py:49: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  den = np.trapz(w, t_common)


np.float64(15.85789830872468)

In [20]:
np.exp(2.75)

np.float64(15.642631884188171)

In [12]:
-np.log(0.016327457207079593)

np.float64(4.114907097101655)

In [9]:
#dir(sampler)
# 'searched_log_rewards_list',
# 'searched_points_list',
np.set_printoptions(precision=15, suppress=False)
sampler.searched_points_list[0][sampler.searched_log_rewards_list[0].argmax()],sampler.searched_log_rewards_list[0].max(),sampler.searched_log_rewards_list[0].argmax()

(array([0.499781468991342, 0.499714682204108, 0.500022033439619,
        0.500080320287737, 0.499971986410774]),
 np.float64(41.825904159834614),
 np.int64(1736))

In [31]:
np.set_printoptions(precision=15, suppress=False)
sampler.searched_points_list[0][sampler.searched_log_rewards_list[0].argmax()].reshape(1,5)

array([[0.500020369896442, 0.500054076830051, 0.499947490236756,
        0.500059958859324, 0.499981864522194]])

In [30]:
log_reward(prior_transform(sampler.searched_points_list[0][-1+sampler.searched_log_rewards_list[0].argmax()].reshape(1,5)))

/tmp/ipykernel_4040671/2888274576.py:48: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  num = np.trapz(w * dphi**2, t_common)
/tmp/ipykernel_4040671/2888274576.py:49: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  den = np.trapz(w, t_common)


array([40.48314478])

In [ ]:
(array([0.499781468991342, 0.499714682204108, 0.500022033439619,
        0.500080320287737, 0.499971986410774]),
 np.float64(41.825904159834614),

In [ ]:
(array([0.50002825, 0.50005415, 0.49995162, 0.50005497, 0.49998339]),
 np.float64(41.440933114624556)) #retro EMRI

In [ ]:
(array([0.49733458, 0.49138475, 0.54312794, 0.50483857, 0.4983772 ]),
 np.float64(42.94630005963107)) #IMRI

In [ ]:
(array([0.50817011, 0.51369668, 0.54436317, 0.50488048, 0.49835934]),
 np.float64(43.01542980501006))

In [17]:
def compute_omega_phi_difference_output(theta, Omega_phi_2PA):
    """
    Compare Omega_phi between 0PA and 2PA trajectories
    
    Args:
        theta: [m1, m2, a, p0, e0, Y0] for 0PA trajectory
        theta_inj_2PA: [m1, m2, a, p0, e0, Y0] for 2PA trajectory  
        dt: time step in seconds
        T: observation time in years
        chi2: secondary spin parameter
        
    Returns:
        scalar difference between interpolated Omega_phi trajectories
    """
    
    # Unpack parameters
    m1_0pa, m2_0pa, a_0pa, p0_0pa, e0_0pa = theta
    
    
    evolve_1PA = False  
    evolve_primary = False  
    evolve_2PA = False  
    add_args = [chi2, evolve_1PA, evolve_primary, evolve_2PA]    
    
    # Generate 0PA trajectory
    traj_0PA = SK_traj_0PA.get_inspiral(
        m1_0pa, m2_0pa, a_0pa, p0_0pa, e0_0pa, Y0, *add_args,
        T=T, dt=dt, err=1e-11, DENSE_STEPPING=False,
        buffer_length=1000, integrate_backwards=False,
        max_step_size=None
    )
    

    # Extract trajectory components
    # Format: (t, p, e, x, Phi_phi, Phi_theta, Phi_r)
    t_0PA, p_0PA, e_0PA, x_0PA = traj_0PA[0], traj_0PA[1], traj_0PA[2], traj_0PA[3]
  
    
    # Calculate fundamental frequencies using geodesic functions
    Omega_phi_0PA, Omega_theta_0PA, Omega_r_0PA = get_fundamental_frequencies(
        a_0pa, p_0PA, e_0PA, x_0PA
    )
    print('Omega_phi_0PA, Omega_theta_0PA, Omega_r_0PA',Omega_phi_0PA[-1], Omega_theta_0PA[-1], Omega_r_0PA[-1])
    print('Omega_phi_2PA, Omega_theta_2PA, Omega_r_2PA',Omega_phi_2PA[-1], Omega_theta_2PA[-1], Omega_r_2PA[-1])

    # Create common time grid for interpolation
    t_min = max(t_0PA.min(), t_2PA.min())
    t_max = min(t_0PA.max(), t_2PA.max())
    t_common = np.linspace(t_min, t_max, 1000)
    
    # Interpolate Omega_phi onto common time grid
    from scipy.interpolate import CubicSpline
    
    # Create splines for both trajectories
    spline_0PA = CubicSpline(t_0PA, Omega_phi_0PA)
    spline_2PA = CubicSpline(t_2PA, Omega_phi_2PA)
    
    # Evaluate on common time grid
    Omega_phi_0PA_interp = spline_0PA(t_common)
    Omega_phi_2PA_interp = spline_2PA(t_common)
    
    # Calculate difference (RMS or mean absolute difference)
    diff_array = Omega_phi_0PA_interp - Omega_phi_2PA_interp
    
    # Return scalar difference (RMS)
    scalar_diff = np.sqrt(np.mean(diff_array**2))
    
    return scalar_diff
    
compute_omega_phi_difference_output(prior_transform(np.array([0.49175409, 0.47710209, 0.54355586, 0.50396805, 0.49869034])),Omega_phi_2PA)

Omega_phi_0PA, Omega_theta_0PA, Omega_r_0PA 0.02989936882805343 0.0283822299286414 0.023501645578410422
Omega_phi_2PA, Omega_theta_2PA, Omega_r_2PA 0.029900709877763002 0.028392566178689852 0.02346986773330028


np.float64(1.243826202939667e-07)